In [8]:
import pytest
import ipytest
from energy import *

In [9]:
ipytest.autoconfig()

In [11]:
%%ipytest

@pytest.fixture
def mock_phy_net_robust():
    class MockPhy:
        def __init__(self):
            self.flag_pv = np.array([False, True]) 
            class MockSNM:
                def __init__(self):
                    self.no_sensors = 2
                    class MockElements:
                        def __init__(self):
                            self.p = np.array([[10.0], [0.05]])
                    self.OTx_elements = MockElements()
                    self.RFTx_elements = MockElements()
            self.snm = MockSNM()
            class MockPVX:
                def __init__(self):
                    self.V = np.array([[0.5]]) 
                    self.I = np.array([[0.02]]) 
                    self.ind = np.array([0])
            self.pvx = MockPVX()
            
    return MockPhy() 

@pytest.fixture
def energy_design_robust():
    """Comprehensive design with mixed uplink types."""
    return {
        'nodes': {
            'sensors': {
                'uplink_type': np.array([1, 0]) # Node 0: RF, Node 1: IR
            }
        },
        'energy_profile': {
            'hardware': {
                'voltage': 3.3, 
                'I_mcu': 2e-3, 
                'I_adc': 1e-3, 
                'I_act': 1.5e-3,
                'I_ext': 0.5e-3,
                'I_sleep': 10e-6, 
                'I_wake': 5e-3
            },
            'tasks': {
                'N_s_up': 1000, 
                'N_c_up': 50000, 
                'L_up_bits': 2048, 
                'L_dw_bits': 512
            },
            'battery': {
                'battery_capacity_mAh': 500, 
                'V_batt': 3.7, 
                'initial_soc': 1.0
            }
        },
        'protocol': {
            'T_cycle': 10, 
            'bit_rate_up_ir': 1e6, 
            'bit_rate_up_rf': 250e3,
            'bit_rate_dw': 100e3,
            't_init': 0.01, 
            't_wait': 0.02,
            'harvesting_hours': 5.0
        }
    }

# --- 1. Driver Integration Tests ---

def test_energy_manager_driver_integration(mock_phy_net_robust, energy_design_robust):
    """Verify that EnergyManager correctly calls IRdriver and RF_calc_I."""
    em = EnergyManager(mock_phy_net_robust, energy_design_robust)
    em.calc_cycle_energy()
    
    # Verify Node 0 (RF) current: 0.24 * 10dBm + 8.8 = 11.2mA
    expected_rf_I = (0.24 * 10.0 + 8.8) * 1e-3
    assert em.I_tx[0] == pytest.approx(expected_rf_I)
    
    # Verify Node 1 (IR) current: Uses the IRdriver polynomial
    # At 0.05W, current should be positive and reasonable
    assert em.I_tx[1] > 2e-3 

# --- 2. Temporal Logic Tests ---

def test_duty_cycle_durations(mock_phy_net_robust, energy_design_robust):
    """Ensure transmission times (d_tx) match the bit rates for different technologies."""
    em = EnergyManager(mock_phy_net_robust, energy_design_robust)
    em.calc_cycle_energy()
    
    # RF Node (Node 0): 2048 bits / 250kbps = 0.008192s
    assert em.d_tx[0] == pytest.approx(2048 / 250000)
    
    # IR Node (Node 1): 2048 bits / 1Mbps = 0.002048s
    assert em.d_tx[1] == pytest.approx(2048 / 1000000)

# --- 3. Harvesting & Sustainability ---

def test_energy_sustainability_report(mock_phy_net_robust, energy_design_robust):
    """Check if the net energy calculation identifies the PV advantage."""
    em = EnergyManager(mock_phy_net_robust, energy_design_robust)
    em.calc_harv_energy(mpp_eff=0.85)
    em.calc_battery_life()
    
    # Net energy calculation: E_harvested - E_consumed
    # Node 1 (PV) should have significantly better net energy than Node 0 (RF)
    assert em.E_day_net[1] > em.E_day_net[0]
    
    # Check that initial battery charge is scaled correctly (mAh * V * 3.6)
    # 500mAh * 3.7V * 3.6 = 6660 Joules
    assert em.batt_charge[0] == pytest.approx(6660.0)

# --- 4. Edge Case: Battery Drain ---

def test_infinite_lifetime_logic(mock_phy_net_robust, energy_design_robust):
    """Verify that if net energy is positive, lifetime is Infinity."""
    em = EnergyManager(mock_phy_net_robust, energy_design_robust)
    
    # Mock a scenario where harvesting is massive
    em.E_day_net = np.array([-100, 500]) # Node 0 draining, Node 1 charging
    em.calc_battery_life()
    
    assert em.days_to_empty[0] != np.inf
    assert em.days_to_empty[1] == np.inf

FFFF                                                                                         [100%]
============================================= FAILURES =============================================
______________________________ test_energy_manager_driver_integration ______________________________

mock_phy_net_robust = <__main__.mock_phy_net_robust.<locals>.MockPhy object at 0x7f79f057b2d0>
energy_design_robust = {'energy_profile': {'battery': {'V_batt': 3.7, 'battery_capacity_mAh': 500, 'initial_soc': 1.0}, 'hardware': {'I_act':...}}, 'protocol': {'T_cycle': 10, 'bit_rate_dw': 100000.0, 'bit_rate_up_ir': 1000000.0, 'bit_rate_up_rf': 250000.0, ...}}

    def test_energy_manager_driver_integration(mock_phy_net_robust, energy_design_robust):
        """Verify that EnergyManager correctly calls IRdriver and RF_calc_I."""
        em = EnergyManager(mock_phy_net_robust, energy_design_robust)
>       em.calc_cycle_energy()

/tmp/ipykernel_759613/2339472182.py:71: 
_ _ _ _ _ _ _ _ _ _ _ _